In [ ]:
import os
import sys
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ----- Configuration -----
REPO_ROOT = "/home/iec/MinhHieu/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
if not os.path.exists(REPO_ROOT):
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

from neural_methods.model.PhysNet import PhysNet_padding_Encoder_Decoder_MAX
from neural_methods.loss.PhysNetNegPearsonLoss import Neg_Pearson

PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupC")
OUTPUT_WEIGHTS_DIR = os.path.join(REPO_ROOT, "final_model_release")
OUTPUT_WEIGHTS_PATH = os.path.join(OUTPUT_WEIGHTS_DIR, "GroupC_PhysNet.pth")

os.makedirs(OUTPUT_WEIGHTS_DIR, exist_ok=True)

CHUNK_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Weights will be saved to: {OUTPUT_WEIGHTS_PATH}")

In [ ]:
# ----- Dataset -----
class PhysNetDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)

        # NDHWC -> NCDHW: transpose (3, 0, 1, 2)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id

In [ ]:
# ----- DataLoader -----
all_input_files = glob.glob(os.path.join(PREPROCESSED_PATH, "*", "*_input*.npy"))
dataset = PhysNetDataset(all_input_files)
print(f"Total clips in dataset: {len(dataset)}")

# Split dataset into 80% train and 20% validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

In [ ]:
# ----- Model, Loss, Optimizer -----
model = PhysNet_padding_Encoder_Decoder_MAX(frames=CHUNK_LENGTH).to(DEVICE)
loss_fn = Neg_Pearson()
optimizer = optim.Adam(model.parameters(), lr=LR)

# OneCycleLR Scheduler as used in PhysnetTrainer
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=LR, 
    epochs=EPOCHS, 
    steps_per_epoch=len(train_loader)
)

In [ ]:
# ----- Training Loop -----
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    print(f"\n==== Training Epoch: {epoch + 1}/{EPOCHS} ====")
    
    # Train
    model.train()
    train_loss = 0.0
    tbar = tqdm(train_loader, desc=f"Train Epoch {epoch + 1}", ncols=80)
    for batch in tbar:
        data = batch[0].to(torch.float32).to(DEVICE)
        label = batch[1].to(torch.float32).to(DEVICE)
        
        rPPG, x_visual, x_visual3232, x_visual1616 = model(data)
        
        # Normalize as in PhysnetTrainer
        rPPG = (rPPG - torch.mean(rPPG)) / torch.std(rPPG)
        label = (label - torch.mean(label)) / torch.std(label)
        
        loss = loss_fn(rPPG, label)
        loss.backward()
        
        train_loss += loss.item()
        tbar.set_postfix(loss=f"{loss.item():.4f}")
        
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0.0
    vbar = tqdm(val_loader, desc=f"Val Epoch {epoch + 1}", ncols=80)
    with torch.no_grad():
        for batch in vbar:
            data = batch[0].to(torch.float32).to(DEVICE)
            label = batch[1].to(torch.float32).to(DEVICE)
            
            rPPG, x_visual, x_visual3232, x_visual1616 = model(data)
            
            # Normalize
            rPPG = (rPPG - torch.mean(rPPG)) / torch.std(rPPG)
            label = (label - torch.mean(label)) / torch.std(label)
            
            loss = loss_fn(rPPG, label)
            val_loss += loss.item()
            vbar.set_postfix(loss=f"{loss.item():.4f}")
            
    val_loss /= len(val_loader)
    print(f"Epoch {epoch + 1} Summary - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Save Best Model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), OUTPUT_WEIGHTS_PATH)
        print(f"Update best model! Saved to: {OUTPUT_WEIGHTS_PATH} (Loss: {best_val_loss:.4f})")

print("\nTraining Completed!")
print(f"Best Validation Loss: {best_val_loss:.4f}")